# Session 2 · Part 1 — Prepare DGAT inputs

**Independent checkpoint:** load and align the official RNA, ADT, and spatial observations. Run the asset downloader first; this part intentionally does not substitute synthetic data.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


In [ ]:
import pandas as pd

from dgat_tutorial.data import load_tutorial_data

dataset = load_tutorial_data(paths.raw_data, allow_demo=False)
spots, transcripts, proteins = dataset.spots, dataset.transcripts, dataset.proteins
common_spots = spots.index.intersection(transcripts.index).intersection(proteins.index)
if common_spots.empty:
    raise ValueError("RNA, ADT, and spatial tables have no shared observation IDs.")

aligned_ids = pd.DataFrame({"spot_id": common_spots})
summary = pd.DataFrame([{
    "aligned_spots": len(common_spots),
    "genes": transcripts.shape[1],
    "proteins": proteins.shape[1],
}])
summary


In [ ]:
ids_path = paths.processed_data / "aligned_spot_ids.csv"
summary_path = paths.results / "session02_input_summary.csv"
aligned_ids.to_csv(ids_path, index=False)
summary.to_csv(summary_path, index=False)
manifest = write_checkpoint(
    "2.1", [ids_path, summary_path], summary=summary.iloc[0].to_dict(), start=paths.root
)
print(f"Checkpoint written: {manifest}")


## Checkpoint

The aligned ID list is the hand-off boundary between data preparation and prediction loading.